# Reading New Df

In [0]:
from pyspark.sql import Row

data_new = [
    Row(product_id=101, product_name="Laptop", category="Electronics", price=52000),
    Row(product_id=102, product_name="Mouse", category="Electronics", price=500),
    Row(product_id=103, product_name="Chair", category="Furniture", price=2000),
    Row(product_id=104, product_name="Desk", category="Furniture", price=7000),
    Row(product_id=105, product_name="Monitor", category="Electronics", price=12000),
    Row(product_id=101, product_name="Laptop", category="Electronics", price=52000),
    Row(product_id=104, product_name="Desk", category="Furniture", price=7000),
]

df_new = spark.createDataFrame(data_new)
display(df_new)

# Reading Old Df

In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import current_timestamp
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, BooleanType

data_old = [
    Row(product_id=101, product_name="Laptop", category="Electronics", price=50000,
        start_date="2024-01-01", end_date=None, is_current=True),

    Row(product_id=102, product_name="Mouse", category="Electronics", price=450,
        start_date="2024-01-01", end_date=None, is_current=True),

    Row(product_id=103, product_name="Chair_old", category="Furniture", price=2000,
        start_date="2024-01-01", end_date=None, is_current=True)
]

schema = StructType([
    StructField("product_id", IntegerType(), True),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("price", IntegerType(), True),
    StructField("start_date", StringType(), True),
    StructField("end_date", StringType(), True),  # explicitly defined
    StructField("is_current", BooleanType(), True)
])

df_old = spark.createDataFrame(data_old, schema=schema)

display(df_old)

# Join New and Old

In [0]:
df_join = df_new.alias('new').join(df_old.alias('old'), on='product_id', how='left')

df_join.display()

In [0]:
from pyspark.sql.functions import col

df_changed = df_join.filter((col('old.product_id').isNotNull()) & 
                            (col('old.product_name') != col('new.product_name')) |
                            (col('old.category') != col('new.category')) |
                            (col('old.price') != col('new.price')))

display(df_changed)

# Expire Old Records

In [0]:
df_changed.display()

In [0]:
from pyspark.sql.functions import lit, current_timestamp, current_date

df_expired = df_changed.select('old.*')\
                        .withColumn('is_current', lit(False))\
                        .withColumn('end_date', current_timestamp())

df_expired.display()

In [0]:

df_join_brand_new = df_new.alias('new').join(df_old.alias('old'), on='product_id', how='leftanti')

df_join_brand_new.display()

# SCD Type 1 & SCD Type 2 Without Join

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
df_new.display()

df_old.display()

# To check duplicate present or not

In [0]:
df_new.groupBy(df_new.columns).count().filter(col('count')>1).orderBy(col('product_id').desc()).display()

## Removing Duplicates from Source Df

In [0]:
df_new = df_new.withColumn('rn', row_number().over(Window.partitionBy(col('product_id')).orderBy(col('price').desc()))).filter(col('rn') == 1).drop(col('rn'))
df_new.display()


In [0]:
df_new.groupBy(df_new.columns).count().filter(col('count')>1).orderBy(col('product_id').desc()).display()

In [0]:
# ═══════════════════════════════════════════════════════════════════════════
#  SCD TYPE 1 + TYPE 2  |  Gold Dim Products
#  No DataFrame join — uses Delta MERGE directly against source
#  Two-MERGE pattern to avoid multiple-source-row conflict
# ═══════════════════════════════════════════════════════════════════════════

from pyspark.sql.functions import (
    col, current_timestamp, lit, row_number, when
)
from pyspark.sql.window import Window
from pyspark.sql.types import TimestampType, BooleanType
from delta.tables import DeltaTable

TARGET_TABLE  = "medallion_arc_e2e.gold.dim_products"
SOURCE_TABLE  = "medallion_arc_e2e.silver.products"

# SCD2 → version history tracked  |  SCD1 → overwrite in-place
SCD2_COLS = ["product_name", "category", "brand"]
SCD1_COLS = ["price", "discounted_price", "stock"]
BUSINESS_KEY = "product_id"

In [0]:
if not spark.catalog.tableExists(TARGET_TABLE):
    # ── First run — create table directly ───────────────────────────────────
    (
        df_new
        .withColumn("is_current", lit(True).cast(BooleanType()))
        .withColumn("start_date", current_timestamp())
        .withColumn("end_date",   lit(None).cast(TimestampType()))
        .withColumn("updated_at", current_timestamp())
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(TARGET_TABLE)
    )
    print(f"[INIT] Gold table created: {TARGET_TABLE}")

else:
    # ── Subsequent runs — MERGE logic ────────────────────────────────────────

    # STEP 3: Register source as temp view
    df_new.withColumn("updated_at", current_timestamp()) \
             .createOrReplaceTempView("products_source")

    delta_table = DeltaTable.forName(spark, TARGET_TABLE)

    # STEP 4: MERGE 1 — Expire SCD2 + Overwrite SCD1
    (
        delta_table.alias("t")
        .merge(
            source    = spark.table("products_source").alias("s"),
            condition = f"t.{BUSINESS_KEY} = s.{BUSINESS_KEY} AND t.is_current = true"
        )
        .whenMatchedUpdate(
            condition = """
                t.product_name <> s.product_name OR
                t.category     <> s.category     OR
                t.brand        <> s.brand
            """,
            set = {
                "is_current": "false",
                "end_date":   "s.updated_at",
                "updated_at": "s.updated_at"
            }
        )
        .whenMatchedUpdate(
            condition = """
                t.product_name  = s.product_name AND
                t.category      = s.category     AND
                t.brand         = s.brand         AND
                (
                    t.price            <> s.price            OR
                    t.discounted_price <> s.discounted_price OR
                    t.stock            <> s.stock
                )
            """,
            set = {
                "price":            "s.price",
                "discounted_price": "s.discounted_price",
                "stock":            "s.stock",
                "updated_at":       "s.updated_at"
            }
        )
        .execute()
    )
    print("[MERGE 1] SCD2 rows expired + SCD1 rows updated in-place")

    # STEP 5: MERGE 2 — Insert new rows  ← still inside else ✅
    (
        delta_table.alias("t")
        .merge(
            source    = spark.table("products_source").alias("s"),
            condition = f"t.{BUSINESS_KEY} = s.{BUSINESS_KEY} AND t.is_current = true"
        )
        .whenNotMatchedInsert(
            values = {
                "product_id":       "s.product_id",
                "product_name":     "s.product_name",
                "category":         "s.category",
                "brand":            "s.brand",
                "price":            "s.price",
                "discounted_price": "s.discounted_price",
                "stock":            "s.stock",
                "is_current":       "true",
                "start_date":       "s.updated_at",
                "end_date":         "null",
                "updated_at":       "s.updated_at"
            }
        )
        .execute()
    )
    print("[MERGE 2] New products + new SCD2 versions inserted")

# ── STEP 6: Validate — runs regardless of if/else path ──────────────────────
df_gold = spark.read.table(TARGET_TABLE)
# ... assertions ...